In [1]:
import kagglehub
import pandas as  pd  
import os
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense

/workspaces/Deep-Learning/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
I0000 00:00:1789973449.796246   35891 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1789973450.771176   35891 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1789973454.401233   35891 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
path = kagglehub.dataset_download("jamaltariqcheema/pima-indians-diabetes-dataset")
df = pd.read_csv(os.path.join(path, "diabetes.csv"))
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72.0,35,169.5,33.6,0.627,50,1
1,1,85,66.0,29,102.5,26.6,0.351,31,0
2,8,183,64.0,32,169.5,23.3,0.672,32,1
3,1,89,66.0,23,94.0,28.1,0.167,21,0
4,0,137,40.0,35,168.0,43.1,2.288,33,1


In [3]:
df.corr()['Outcome']

Pregnancies                 0.221898
Glucose                     0.495990
BloodPressure               0.174469
SkinThickness               0.295138
Insulin                     0.377081
BMI                         0.315577
DiabetesPedigreeFunction    0.173844
Age                         0.238356
Outcome                     1.000000
Name: Outcome, dtype: float64

In [4]:
x = df.iloc[:,0:-1].values
y = df.iloc[:,-1].values


In [5]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

scaler = StandardScaler()
x = scaler.fit_transform(x)
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 1)

In [6]:

model = Sequential()
model.add(Dense(32, activation="relu", input_dim = 8))
model.add(Dense(1, activation='sigmoid'))

model.compile(optimizer='Adam', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(x_train, y_train, batch_size=32, epochs = 100, validation_data=(x_test, y_test))

Epoch 1/100


/workspaces/Deep-Learning/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
E0000 00:00:1789973458.939039   35891 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.4853 - loss: 0.7258 - val_accuracy: 0.5909 - val_loss: 0.6845
Epoch 2/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6612 - loss: 0.6497 - val_accuracy: 0.6883 - val_loss: 0.6200
Epoch 3/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7182 - loss: 0.5931 - val_accuracy: 0.7338 - val_loss: 0.5701
Epoch 4/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7508 - loss: 0.5527 - val_accuracy: 0.7792 - val_loss: 0.5312
Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7785 - loss: 0.5214 - val_accuracy: 0.7857 - val_loss: 0.5017
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7834 - loss: 0.4992 - val_accuracy: 0.7987 - val_loss: 0.4816
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7915 - loss: 0.4823 - val_accuracy: 0.8117 - val_loss: 0.4644
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7964 - loss: 0.4688 - val_accuracy: 0.8247 - val_loss: 0.

<h3>But this architecture was purely based on the intution</h3>
But actually the question is how to select:
<ol>
<li>An appropiate optimizer
<li>Number of nodes in a layer
<li>Number of hidden layers
</ol>

In [7]:
import kerastuner as kt

/tmp/ipykernel_35891/1654478174.py:1: DeprecationWarning: `import kerastuner` is deprecated, please use `import keras_tuner`.
  import kerastuner as kt


<h1>Selecting an appropiate Optimizer</h1>

We makea function which takes 'hp' and tunes it according to choice given

In [15]:
def build_model(hp):
    model = Sequential()
    model.add(Dense(32, activation = "relu", input_dim = 8))
    model.add(Dense(1, activation="sigmoid"))

    optimizer = hp.Choice('optimizer', values =['adam', 'sgd', 'rmsprop', 'adadelta'])
    model.compile(optimizer = optimizer, loss = 'binary_crossentropy', metrics = ['accuracy'])

    return model

We make a tuner object which is bacially a keras tuner object which have an 'objective'<br>
It does a random search as in it chooses different combinations of hyperparameters randomly to test<br>
Since we have given max_trails = 5 it will try 5 different combinations<br>
Now the directiry part is optional as it if we dont use it, it bydefault creates json files for each trail<br>
<br>
kerastuner.RandomSearch( HYPERMODEL, OBJECTIVE, MAX_TRIAL, SEED)

In [18]:
import tempfile
tuner = kt.RandomSearch(build_model, objective = 'val_accuracy',directory = tempfile.mkdtemp(), max_trials = 5)


This is the actual serach function which does the experiment

In [19]:
tuner.search(x_train, y_train, epochs = 5, validation_data = (x_test, y_test))

Trial 4 Complete [00h 00m 02s]
val_accuracy: 0.7727272510528564

Best val_accuracy So Far: 0.850649356842041
Total elapsed time: 00h 00m 07s


In [23]:
#Thsi gives summary of each trial
tuner.results_summary()

Results summary
Results in /tmp/tmpx3mxvkiw/untitled_project
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 2 summary
Hyperparameters:
optimizer: adam
Score: 0.850649356842041

Trial 0 summary
Hyperparameters:
optimizer: rmsprop
Score: 0.8051947951316833

Trial 3 summary
Hyperparameters:
optimizer: sgd
Score: 0.7727272510528564

Trial 1 summary
Hyperparameters:
optimizer: adadelta
Score: 0.649350643157959


In [25]:
#Gives the best hyperparameter
tuner.get_best_hyperparameters()[0].values

{'optimizer': 'adam'}

Now we can directly train the model using the best hyperparameter

In [26]:
model = tuner.get_best_models(num_models=1)[0]

/workspaces/Deep-Learning/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/workspaces/Deep-Learning/.venv/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:869: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(store)


In [27]:
model.fit(x_train, y_train, epochs = 100, initial_epoch = 6, validation_data=(x_test, y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.7818 - loss: 0.4800 - val_accuracy: 0.8442 - val_loss: 0.4598
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7899 - loss: 0.4586 - val_accuracy: 0.8312 - val_loss: 0.4409
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7899 - loss: 0.4447 - val_accuracy: 0.8312 - val_loss: 0.4270
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7883 - loss: 0.4348 - val_accuracy: 0.8247 - val_loss: 0.4181
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7915 - loss: 0.4284 - val_accuracy: 0.8247 - val_loss: 0.4100
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7915 - loss: 0.4227 - val_accuracy: 0.8247 - val_loss: 0.4058
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7932 - loss: 0.4181 - val_accuracy: 0.8247 - val_loss: 0.4014
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7980 - loss: 0.4142 - val_accuracy: 0.82

<h1>Selecting an appropiate number of nodes in a layer</h1>

In [30]:
def build_model(hp):
    model = Sequential()
    units = hp.Int('units', min_value = 8, max_value = 128, step = 8)

    model.add(Dense(units = units, activation = 'relu', input_dim = 8))
    model.add(Dense(1, activation = 'sigmoid'))
    model.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics=['accuracy'])

    return model

In [31]:
tuner = kt.RandomSearch(build_model, objective = 'val_accuracy', max_trials = 5)

/workspaces/Deep-Learning/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [32]:
tuner.search(x_train, y_train, epochs = 5, validation_data = (x_test, y_test))

Trial 5 Complete [00h 00m 02s]
val_accuracy: 0.8246753215789795

Best val_accuracy So Far: 0.8376623392105103
Total elapsed time: 00h 00m 11s


In [34]:
tuner.get_best_hyperparameters()[0].values

{'units': 72}

In [35]:
model = tuner.get_best_models(num_models=1)[0]
model.fit(x_train, y_train, epochs = 100, initial_epoch = 6, validation_data=(x_test, y_test))

Epoch 7/100


/workspaces/Deep-Learning/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/workspaces/Deep-Learning/.venv/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:869: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(store)


20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.7866 - loss: 0.4545 - val_accuracy: 0.8312 - val_loss: 0.4079
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7948 - loss: 0.4432 - val_accuracy: 0.8377 - val_loss: 0.3972
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7997 - loss: 0.4357 - val_accuracy: 0.8377 - val_loss: 0.3912
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7997 - loss: 0.4296 - val_accuracy: 0.8377 - val_loss: 0.3887
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7997 - loss: 0.4249 - val_accuracy: 0.8377 - val_loss: 0.3851
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8046 - loss: 0.4196 - val_accuracy: 0.8377 - val_loss: 0.3796
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8029 - loss: 0.4146 - val_accuracy: 0.8377 - val_loss: 0.3758
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8078 - loss: 0.4102 - val_accuracy: 0.8377 - val_los